# EEG_07b — Build PyTorch Graph Datasets

Costruisce dataset PyG completi `Data(x, edge_index, y, subj)` per tutti i trial,
per ogni metodo di costruzione del grafo (PCC, PLV, wPLI).

**Esegui una volta sola.** EEG_08/09/10 caricheranno il dataset da disco.

| File output | Contenuto |
|---|---|
| `data/interim/graphs/dataset_pcc_k6.pt` | lista di `Data` con grafi PCC |
| `data/interim/graphs/dataset_plv_k6.pt` | lista di `Data` con grafi PLV |
| `data/interim/graphs/dataset_wpli_k6.pt` | lista di `Data` con grafi wPLI |

**`x`** è salvato grezzo (non normalizzato). La normalizzazione per-canale
viene applicata in EEG_08 usando le statistiche del training set.

**`y`** è `label_idx` grezzo (0-109). EEG_08 applica il mapping cluster in `__getitem__`.

In [ ]:
from pathlib import Path
import numpy as np
import torch
import h5py
import pandas as pd
from tqdm import tqdm
from scipy.signal import butter, filtfilt
from scipy.signal import hilbert as sp_hilbert
from torch_geometric.data import Data

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)

META_CSV   = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH  = project_root / "src" / "io" / "ebneuro.locs"
GRAPHS_DIR = project_root / "data" / "interim" / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

# ── Deve essere identico a EEG_08 ──────────────────────────
K_GRAPH   = 6
SFREQ     = 256
N_CHANS   = 59
PLV_BANDS = [(4, 8), (8, 13)]            # theta + alpha
# PLV_BANDS = [(4, 8), (8, 13), (13, 30), (30, 80)]  # + beta + gamma

METHODS = ["pcc", "plv", "wpli"]

print(f"project_root : {project_root}")
print(f"Output dir   : {GRAPHS_DIR}")
print(f"K_GRAPH      : {K_GRAPH}  |  PLV_BANDS: {PLV_BANDS}")
print(f"Metodi       : {METHODS}")

In [ ]:
meta = pd.read_csv(META_CSV)

# Rimuovi epoche corrotte
meta = meta[~(
    (meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_04.h5") & (meta["epoch_idx"] >= 34))
)].copy()
# Rimuovi righe con subject_id non numerico (es. 'ignore_43')
meta = meta[pd.to_numeric(meta["subject_id"], errors="coerce").notna()].copy()
meta["subject_id"] = meta["subject_id"].astype(int).astype(str).str.zfill(2)

def read_eloc_names(path):
    names = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]

ch_names_61 = read_eloc_names(ELOC_PATH)
EXCLUDE     = {"A1", "A2"}
keep_idx    = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
assert len(keep_idx) == N_CHANS

print(f"Trial totali : {len(meta)}")
print(f"Canali       : {len(keep_idx)}")

In [ ]:
# ── Funzioni costruzione grafo (identiche a EEG_08) ─────────

def knn_from_matrix(matrix, k):
    edges = set()
    for i in range(matrix.shape[0]):
        for j in np.argsort(matrix[i])[::-1][:k]:
            edges.add((i, int(j)))
            edges.add((int(j), i))
    src, dst = zip(*sorted(edges))
    return torch.tensor([list(src), list(dst)], dtype=torch.long)


def pcc_to_edge_index(x_np, k=6):
    pcc = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0.0)
    return knn_from_matrix(pcc, k)


def plv_to_edge_index(x_np, k=6, sfreq=256, bands=None):
    if bands is None:
        bands = PLV_BANDS
    nyq = sfreq / 2.0
    plv_sum = np.zeros((x_np.shape[0], x_np.shape[0]), dtype=np.float64)
    for flo, fhi in bands:
        b, a    = butter(4, [flo / nyq, fhi / nyq], btype="band")
        x_filt  = filtfilt(b, a, x_np, axis=1).astype(np.float32)
        analytic = sp_hilbert(x_filt, axis=1)
        exp_phi  = np.exp(1j * np.angle(analytic))
        plv_sum += np.abs(exp_phi @ exp_phi.conj().T) / x_np.shape[1]
    plv_matrix = (plv_sum / len(bands)).astype(np.float32)
    np.fill_diagonal(plv_matrix, 0.0)
    return knn_from_matrix(plv_matrix, k)


def wpli_to_edge_index(x_np, k=6, sfreq=256, bands=None):
    if bands is None:
        bands = PLV_BANDS
    nyq = sfreq / 2.0
    wpli_sum = np.zeros((x_np.shape[0], x_np.shape[0]), dtype=np.float64)
    for flo, fhi in bands:
        b, a     = butter(4, [flo / nyq, fhi / nyq], btype="band")
        x_filt   = filtfilt(b, a, x_np, axis=1).astype(np.float32)
        analytic = sp_hilbert(x_filt, axis=1)
        imag_cs  = np.imag(
            analytic[:, :, np.newaxis] * analytic[np.newaxis, :, :].conj()
        )
        wpli_sum += np.abs(imag_cs.mean(axis=-1)) / (np.abs(imag_cs).mean(axis=-1) + 1e-8)
    wpli_matrix = (wpli_sum / len(bands)).astype(np.float32)
    np.fill_diagonal(wpli_matrix, 0.0)
    return knn_from_matrix(wpli_matrix, k)


GRAPH_FN = {"pcc": pcc_to_edge_index, "plv": plv_to_edge_index, "wpli": wpli_to_edge_index}
print("Funzioni grafo OK")

In [ ]:
# ============================================================
# BUILD DATASET — parallelizzato su tutti i core disponibili
# ============================================================

import multiprocessing as mp
import gc

N_WORKERS = max(1, mp.cpu_count() - 2)
print(f"Worker: {N_WORKERS} / {mp.cpu_count()} core")


def _compute_one(args):
    """Worker: legge HDF5 e calcola edge_index. Ritorna numpy/list puri."""
    r, method, k = args
    import h5py, numpy as np
    from scipy.signal import butter, filtfilt
    from scipy.signal import hilbert as sp_hilbert

    path  = r["path_h5"]
    e_idx = int(r["epoch_idx"])
    subj  = int(r["subject_id"])
    y     = int(r["label_idx"])

    with h5py.File(path, "r") as f:
        x_np = f["data"][e_idx][KEEP_IDX_GLOBAL, :].astype(np.float32)

    def knn_from_matrix(matrix, k):
        edges = set()
        for i in range(matrix.shape[0]):
            for j in np.argsort(matrix[i])[::-1][:k]:
                edges.add((i, int(j))); edges.add((int(j), i))
        return list(zip(*sorted(edges)))

    if method == "pcc":
        mat = np.abs(np.corrcoef(x_np)); np.fill_diagonal(mat, 0.0)
    else:
        nyq = SFREQ / 2.0
        acc = np.zeros((x_np.shape[0], x_np.shape[0]), dtype=np.float64)
        for flo, fhi in PLV_BANDS_GLOBAL:
            b, a     = butter(4, [flo / nyq, fhi / nyq], btype="band")
            x_filt   = filtfilt(b, a, x_np, axis=1).astype(np.float32)
            analytic = sp_hilbert(x_filt, axis=1)
            if method == "plv":
                exp_phi = np.exp(1j * np.angle(analytic))
                acc    += np.abs(exp_phi @ exp_phi.conj().T) / x_np.shape[1]
            else:
                imag_cs = np.imag(analytic[:, :, np.newaxis] * analytic[np.newaxis, :, :].conj())
                acc    += np.abs(imag_cs.mean(axis=-1)) / (np.abs(imag_cs).mean(axis=-1) + 1e-8)
        mat = (acc / len(PLV_BANDS_GLOBAL)).astype(np.float32)
        np.fill_diagonal(mat, 0.0)

    src_dst = knn_from_matrix(mat, k)
    return (x_np, list(src_dst[0]), list(src_dst[1]), y, subj)


KEEP_IDX_GLOBAL  = keep_idx
PLV_BANDS_GLOBAL = PLV_BANDS

records = meta[["path_h5", "epoch_idx", "label_idx", "subject_id"]].to_dict("records")

for method in METHODS:
    out_path = GRAPHS_DIR / f"dataset_{method}_k{K_GRAPH}.pt"
    if out_path.exists():
        print(f"[{method}] già calcolato → {out_path.name}  (skip)")
        continue

    print(f"\n[{method}] Costruisco dataset ({len(records)} trial) con {N_WORKERS} worker...")
    tasks = [(r, method, K_GRAPH) for r in records]

    with mp.Pool(processes=N_WORKERS) as pool:
        results = list(tqdm(
            pool.imap(_compute_one, tasks, chunksize=64),
            total=len(tasks), desc=method
        ))

    print(f"  Assemblaggio Data objects...")
    data_list = [
        Data(
            x          = torch.tensor(x_np, dtype=torch.float32),
            edge_index = torch.tensor([src, dst], dtype=torch.long),
            y          = torch.tensor(y,    dtype=torch.long),
            subj       = torch.tensor(subj, dtype=torch.long),
        )
        for x_np, src, dst, y, subj in results
    ]

    # Libera results prima di salvare (evita doppio picco di memoria)
    del results
    gc.collect()

    torch.save(data_list, out_path)
    size_gb = out_path.stat().st_size / 1e9
    print(f"[{method}] Salvato: {out_path.name}  ({len(data_list)} grafi, {size_gb:.2f} GB)")

    # Libera data_list prima del prossimo metodo
    del data_list
    gc.collect()
    print(f"  Memoria liberata. Prossimo metodo...")

print("\nBuild completato.")


In [ ]:
# ── Verifica ─────────────────────────────────────────────────
print(f"File in {GRAPHS_DIR}:")
for method in METHODS:
    out_path = GRAPHS_DIR / f"dataset_{method}_k{K_GRAPH}.pt"
    if out_path.exists():
        dl = torch.load(out_path, weights_only=False)
        d0 = dl[0]
        size_gb = out_path.stat().st_size / 1e9
        print(f"  {out_path.name:<35} {len(dl):>6} grafi  {size_gb:.2f} GB")
        print(f"    x: {tuple(d0.x.shape)}  edge_index: {tuple(d0.edge_index.shape)}  y: {d0.y.item()}  subj: {d0.subj.item()}")
    else:
        print(f"  {out_path.name} — NON TROVATO")